# Stage 2 joint experiment notebook

Run order after Exp2 refactor: 00 prepare dataset → 01 Exp2A baseline → 02 Exp2B GCA baseline → 03 Exp2D lane detail neck → 04 Exp2E matched lane loss → 05 Exp2F detail + matching. Run 06 KD only if the teacher checkpoint exists. Run Exp3 notebooks only after Exp2F improves lane geometry. Every notebook re-extracts the Drive tar into `/content`; never assume files from a previous Colab runtime still exist. Training logs are printed in this notebook cell and mirrored to `/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/`.


# Stage 2 Notebook 07 - Loss weight sweep and seed stability

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)


Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
import itertools, subprocess, sys, yaml, tempfile
from pathlib import Path

BASE_CONFIG = Path('stage2/configs/exp02_rmt_gca_joint.yaml')
seeds = [0, 1, 2]
lambda_values = [0.25, 0.5, 1.0, 2.0]
for seed, lam in itertools.product(seeds, lambda_values):
    cfg = yaml.safe_load(BASE_CONFIG.read_text())
    cfg['run']['seed'] = seed
    cfg['loss']['lambda_mode'] = 'fixed'
    cfg['loss']['lambda_lane'] = lam
    cfg['run']['name'] = f"exp02_sweep_seed{seed}_lambda{lam}"
    cfg['run']['work_dir'] = f"/content/{cfg['run']['name']}"
    cfg['run']['output_tar'] = f"/content/drive/MyDrive/EcoCAR/training_runs/{cfg['run']['name']}.tar"
    tmp = Path('/content') / f"{cfg['run']['name']}.yaml"
    tmp.write_text(yaml.safe_dump(cfg, sort_keys=False))
    cmd = [sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py', '--config', str(tmp), '--limit-train', '1500', '--limit-val', '300', '--print-every', '50']
    print(' '.join(cmd), flush=True)
    run_streaming(cmd, log_path=os.path.join(LOG_DIR, f'{Path(tmp).stem}_train.log'))


流式输出内容被截断，只能显示最后 5000 行内容。
  "val/det/obj": 0.19094115257263183,
  "val/det/cls": 0.0002079546492313966,
  "val/det/bbox_l1": 0.04611347079277039,
  "val/det/giou": 0.8437242245674134,
  "val/det/dn": 0.0,
  "val/det/total": 2.109164905548096,
  "val/det/positives": 85.36
}
[epoch_summary] epoch=6 phase=full_finetune train_total=7.4904 train_det=2.1316 train_lane=2.6794 val_det=2.1092 val_lane=2.6846 val_lane_exist_acc=0.6705 val_lane_point_mae=0.5668 best_score=-7.4617 elapsed_s=29.2 peak_mem_mb=5995
epoch=7 phase=full_finetune batches=187
epoch=7 waiting_for_first_batch...
epoch=7 step=1/187 total=7.3937 det=2.0538 lane=2.6700 lambda_lane=2.0000 grad_cos=0.0097 speed=2.08step/s eta_min=1.5
epoch=7 step=50/187 total=7.2845 det=2.0475 lane=2.6185 lambda_lane=2.0000 grad_cos=nan speed=7.36step/s eta_min=0.3
epoch=7 step=100/187 total=7.5702 det=2.2400 lane=2.6651 lambda_lane=2.0000 grad_cos=nan speed=7.44step/s eta_min=0.2
epoch=7 step=150/187 total=7.4823 det=2.2018 lane=2.6403 lambda_